In [10]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

# ================= 1. DOMAIN LOGIC ===================
TYPE_RELATIONS = {
    "coffee shop": ["cafe", "tea shop", "bakery"],
    "restaurant": ["bar/pub", "vegetarian", "bakery", "food"],
    "bar/pub": ["restaurant"],
    "natural": ["attraction", "entertainment", "park", "beach"],
    "attraction": ["natural", "cultural", "historical"],
    "cultural": ["attraction", "historical", "museum"],
    "historical": ["attraction", "cultural"],
    "hotel": ["villa", "resort", "homestay", "hostel", "apartment"],
    "shopping": ["entertainment", "market", "mall"],
    "entertainment": ["shopping", "natural"],
}

def calculate_similarity_score(user_type, poi_type):
    u = str(user_type).lower().strip()
    p = str(poi_type).lower().strip()

    if u == p: return 0.30
    if p in TYPE_RELATIONS.get(u, []): return 0.15
    return 0.0

# ================= 2. BASE CLASS =====================
class BaseRecommender:
    def __init__(self, model_name="base"):
        self.model = None
        self.model_name = model_name
        self.encoders = {}
        self.scaler = None # Dùng cho Linear Regression nếu cần

        self.cat_features = ["user_city", "user_type", "poi_city", "poi_type"]
        
        # Model train luôn dùng tên 'poi_price', dù dữ liệu gốc là 'price'
        self.num_features = ["user_price", "poi_price", "rating", "latitude", "longitude", "type_match_score"]
        self.features = self.cat_features + self.num_features

    # -------- Feature Engineering ----------
    def feature_engineering(self, df):
        df = df.copy()
        df["type_match_score"] = df.apply(
            lambda x: calculate_similarity_score(x.get("user_type", ""), x.get("poi_type", "")),
            axis=1,
        )
        # Fill NA cho các cột số
        for col in ["user_price", "poi_price", "rating"]:
            if col in df.columns: df[col] = df[col].fillna(0)
        
        # Fill NA cho tọa độ (tránh lỗi Planner)
        for col in ["latitude", "longitude"]:
            if col in df.columns: df[col] = df[col].fillna(0.0)
            
        return df

    # -------- Encoding ---------------------
    def fit_encoders(self, df):
        for col in self.cat_features:
            le = LabelEncoder()
            df[col] = df[col].astype(str).fillna("unknown")
            df[col] = le.fit_transform(df[col])
            self.encoders[col] = le
        return df

    def transform_encoders(self, df):
        df = df.copy()
        for col in self.cat_features:
            if col in self.encoders:
                le = self.encoders[col]
                # Map an toàn: nếu gặp nhãn lạ -> -1
                df[col] = df[col].astype(str).map(
                    lambda s: le.transform([s])[0] if s in le.classes_ else -1
                )
        return df

    # -------- Helpers ---------------------
    def split_data(self, X, y):
        X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
        X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
        return X_train, X_val, X_test, y_train, y_val, y_test

    def evaluate(self, X_test, y_test):
        preds = self.model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        return {
            "MAE": mean_absolute_error(y_test, preds),
            "RMSE": rmse,
            "R2": r2_score(y_test, preds),
        }

    # -------- Save / Load --------------------------
    def save(self, path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        # Lưu cả model, encoder và scaler
        payload = {"model": self.model, "encoders": self.encoders, "scaler": self.scaler}
        with open(path, "wb") as f:
            pickle.dump(payload, f)
        print(f"✓ [{self.model_name}] Model saved to {path}")

    def load(self, path):
        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {path}")
        with open(path, "rb") as f:
            payload = pickle.load(f)
        self.model = payload["model"]
        self.encoders = payload.get("encoders", {})
        self.scaler = payload.get("scaler", None)
        print(f"✓ [{self.model_name}] Model loaded.")

    # -------- RECOMMEND (CORE FUNCTION) ----------
    def recommend(self, df_raw, city, user_type, price, top_k=100):
        """
        Dự đoán và trả về danh sách POI kèm Score và các thông tin cần thiết cho Planner.
        """
        city_norm = city.lower().strip()
        
        # Hỗ trợ user_type là list hoặc string
        if isinstance(user_type, str):
            user_types = [user_type.lower().strip()]
        else:
            user_types = [ut.lower().strip() for ut in user_type]

        # Lọc theo thành phố
        subset = df_raw[df_raw["city_norm"] == city_norm].copy()
        if subset.empty: return pd.DataFrame()

        # Tạo input giả lập (Cartesian Product: User x All POIs)
        rows = []
        for _, poi in subset.iterrows():
            for ut in user_types:
                rows.append({
                    "user_city": city_norm,
                    "user_type": ut,
                    "user_price": price,
                    
                    # Mapping từ dữ liệu gốc (POI.csv)
                    "poi_city": poi.get("city_norm", ""),
                    "poi_type": poi.get("type", "unknown"),

                    # [QUAN TRỌNG] Map 'price_level' (POI) -> 'poi_price' (Model Feature)
                    "poi_price": poi.get("price_level", 0), 
                    
                    "rating": poi.get("rating", 0),
                    "latitude": poi.get("latitude", 0.0),
                    "longitude": poi.get("longitude", 0.0),
                    "poi_id": poi["poi_id"],
                    "name": poi["name"],
                })

        inf = pd.DataFrame(rows)

        # 1. Feature Engineering
        processed = self.feature_engineering(inf)

        # 2. Xử lý Input tùy theo Model Type
        if "CatBoost" in self.model_name:
            # CatBoost cần string, không cần transform số
            for col in self.cat_features:
                processed[col] = processed[col].astype(str).fillna("unknown")
            X_pred = processed[self.features]
        else:
            # Linear/RandomForest cần transform số
            processed = self.transform_encoders(processed)
            X_pred = processed[self.features]
            # Linear cần Scaler
            if self.scaler:
                X_pred = self.scaler.transform(X_pred)

        # 3. Predict Score
        inf["score"] = self.model.predict(X_pred)

        # 4. Gom nhóm & Trả về đủ cột cho Planner/Optimizer
        # (latitude, longitude, poi_type, poi_price là bắt buộc để Planner chạy)
        return (
            inf.groupby("poi_id", as_index=False)
            .agg({
                "name": "first",
                "score": "mean",
                "latitude": "first",
                "longitude": "first",
                "poi_type": "first",
                "poi_price": "first"
            })
            .sort_values("score", ascending=False)
            .head(top_k)
        )

    # Hàm abstract
    def train(self, df):
        raise NotImplementedError


# ================= 3. IMPLEMENTATIONS =================

class LinearRecommender(BaseRecommender):
    def __init__(self):
        super().__init__(model_name="LinearRegression")

    def train(self, df):
        df = self.feature_engineering(df)
        df = self.fit_encoders(df)

        X = df[self.features]
        y = df["label"]
        
        # Linear cần Scaling để tốt hơn
        self.scaler = StandardScaler()
        X = self.scaler.fit_transform(X)

        X_train, X_val, X_test, y_train, y_val, y_test = self.split_data(X, y)

        self.model = LinearRegression()
        self.model.fit(X_train, y_train)

        val_mae = mean_absolute_error(y_val, self.model.predict(X_val))
        test_metrics = self.evaluate(X_test, y_test)
        test_metrics["Val_MAE"] = val_mae
        
        print(f"[{self.model_name}] Train finished. Val MAE: {val_mae:.4f}")
        return test_metrics


class RandomForestRecommender(BaseRecommender):
    def __init__(self):
        super().__init__(model_name="RandomForest")

    def train(self, df):
        df = self.feature_engineering(df)
        df = self.fit_encoders(df)

        X = df[self.features]
        y = df["label"]

        X_train, X_val, X_test, y_train, y_val, y_test = self.split_data(X, y)

        self.model = RandomForestRegressor(
            n_estimators=200, max_depth=12, random_state=42, n_jobs=-1
        )
        self.model.fit(X_train, y_train)

        val_mae = mean_absolute_error(y_val, self.model.predict(X_val))
        test_metrics = self.evaluate(X_test, y_test)
        test_metrics["Val_MAE"] = val_mae
        
        print(f"[{self.model_name}] Train finished. Val MAE: {val_mae:.4f}")
        return test_metrics


class CatBoostRecommender(BaseRecommender):
    def __init__(self):
        super().__init__(model_name="CatBoost")
        # Không cần self.encoders vì CatBoost xử lý nội tại

    def train(self, df):
        df = self.feature_engineering(df)
        
        # CatBoost cần string sạch
        for col in self.cat_features:
            df[col] = df[col].astype(str).fillna("unknown")

        X = df[self.features]
        y = df["label"]

        X_train, X_val, X_test, y_train, y_val, y_test = self.split_data(X, y)

        self.model = CatBoostRegressor(
            iterations=500,
            depth=6,
            learning_rate=0.05,
            loss_function="RMSE",
            eval_metric="MAE",
            verbose=False,
            allow_writing_files=False,
            early_stopping_rounds=50
        )

        self.model.fit(
            X_train,
            y_train,
            cat_features=self.cat_features,
            eval_set=(X_val, y_val),
            use_best_model=True,
        )
        
        self.learning_curve = self.model.get_evals_result()["validation"]["MAE"]
        
        test_metrics = self.evaluate(X_test, y_test)
        print(f"[{self.model_name}] Train finished. Best Iteration: {self.model.get_best_iteration()}")
        return test_metrics
    
    # Không cần override recommend vì BaseRecommender đã xử lý if "CatBoost"

In [11]:
if __name__ == "__main__":
    # --- CẤU HÌNH ĐƯỜNG DẪN ---
    # Hãy sửa lại đường dẫn này cho đúng với máy của bạn
    TRAIN_DATA_PATH = "../data/recommender_training.csv"
    MODEL_DIR = "../models"
    
    # 1. Load Data
    print("Loading training data...")
    try:
        df_train = pd.read_csv(TRAIN_DATA_PATH)
        print(f"Data loaded: {df_train.shape}")
    except FileNotFoundError:
        print(f"ERROR: Không tìm thấy file tại {TRAIN_DATA_PATH}")
        exit()

    # 2. Khởi tạo các model
    models = [
        LinearRecommender(),
        RandomForestRecommender(),
        CatBoostRecommender()
    ]

    results = []

    # 3. Vòng lặp Train & Save
    for model in models:
        print(f"\n--- Training {model.model_name} ---")
        metrics = model.train(df_train)
        metrics["Model"] = model.model_name
        results.append(metrics)
        
        save_path = os.path.join(MODEL_DIR, f"{model.model_name.lower()}_rec.pkl")
        model.save(save_path)

    print("\n✓ ĐÃ HOÀN TẤT TRAIN VÀ LƯU 3 MODEL.")
    print("Bây giờ bạn có thể dùng file .pkl này cho phần Planner.")
    
    print(pd.DataFrame(results))


Loading training data...
Data loaded: (117936, 11)

--- Training LinearRegression ---
[LinearRegression] Train finished. Val MAE: 0.0637
✓ [LinearRegression] Model saved to ../models\linearregression_rec.pkl

--- Training RandomForest ---
[RandomForest] Train finished. Val MAE: 0.0007
✓ [RandomForest] Model saved to ../models\randomforest_rec.pkl

--- Training CatBoost ---
[CatBoost] Train finished. Best Iteration: 499
✓ [CatBoost] Model saved to ../models\catboost_rec.pkl

✓ ĐÃ HOÀN TẤT TRAIN VÀ LƯU 3 MODEL.
Bây giờ bạn có thể dùng file .pkl này cho phần Planner.
        MAE      RMSE        R2   Val_MAE             Model
0  0.063820  0.084625  0.529635  0.063725  LinearRegression
1  0.000661  0.004072  0.998911  0.000739      RandomForest
2  0.000920  0.002416  0.999617       NaN          CatBoost


In [12]:

from preprocess import load_and_clean_data


poi_df = load_and_clean_data("../data/POI.csv")

engine = CatBoostRecommender()
engine.load("../models/catboost_rec.pkl")

top_poi = engine.recommend(
    df_raw=poi_df,
    city="da lat",
    user_type=["restaurant"],
    price=2,
    top_k=10
)

print(top_poi)


✓ [CatBoost] Model loaded.
              poi_id                             name     score   latitude  \
218  restaurant01778                   VLC Restaurant  0.995759  11.944382   
203  restaurant01763              Restaurant Drinkery  0.994571  11.944411   
208  restaurant01768              Com Linh Restaurant  0.993602  11.938524   
204  restaurant01764  Tiệm ăn Đà Lạt Phố - Restaurant  0.991863  11.944284   
236  restaurant01796                    Hawaiian Poke  0.991856  11.941688   
219  restaurant01779       Ty Crêpe French Restaurant  0.991846  11.952894   
213  restaurant01773    MiMi Sushi & Pizza Restaurant  0.989987  11.946473   
211  restaurant01771                KALACÁ Restaurant  0.989977  11.935202   
205  restaurant01765       Trang's Cookery Restaurant  0.989965  11.947187   
206  restaurant01766                    AUSSIE BURGER  0.988540  11.944045   

      longitude    poi_type  poi_price  
218  108.456459  restaurant          1  
203  108.434265  restaurant     

In [13]:
import pandas as pd
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

class RouteOptimizer:
    def __init__(self, duration_file, distance_file, poi_file):
        # Load dữ liệu ma trận
        # fillna(0) để đảm bảo không lỗi tính toán
        self.duration_matrix = pd.read_csv(duration_file, index_col='poi_id').fillna(0)
        self.distance_matrix = pd.read_csv(distance_file, index_col='poi_id').fillna(0)
        
    def get_time_between(self, from_id, to_id):
        try:
            # OR-Tools bắt buộc input là số nguyên (int)
            val = self.duration_matrix.loc[from_id, to_id]
            return int(val)
        except KeyError:
            return 100000 # Phạt nặng nếu không tìm thấy đường

    def optimize_route(self, selected_poi_ids, start_poi_id, max_time_minutes=720, visit_time_per_poi=60):
        """
        Sắp xếp thứ tự đi tối ưu cho một danh sách điểm.
        """
        # 1. Chuẩn bị dữ liệu (Đưa điểm xuất phát lên đầu list)
        targets = [pid for pid in selected_poi_ids if pid != start_poi_id]
        full_ids = [start_poi_id] + targets
        
        # Map: Index (0,1,2) <-> POI_ID (String)
        idx_to_id = {i: pid for i, pid in enumerate(full_ids)}
        n = len(full_ids)
        
        # Tạo ma trận thời gian con (Sub-matrix)
        time_matrix = {}
        for i in range(n):
            time_matrix[i] = {}
            for j in range(n):
                if i == j: 
                    time_matrix[i][j] = 0
                else:
                    travel = self.get_time_between(idx_to_id[i], idx_to_id[j])
                    # Nếu j là điểm đến, cộng thêm thời gian chơi. Về khách sạn (node 0) thì không cộng.
                    visit = 0 if j == 0 else int(visit_time_per_poi)
                    time_matrix[i][j] = travel + visit

        # 2. Cấu hình OR-Tools
        manager = pywrapcp.RoutingIndexManager(n, 1, 0) # 1 xe, xuất phát tại 0
        routing = pywrapcp.RoutingModel(manager)
        
        def time_callback(from_idx, to_idx):
            from_node = manager.IndexToNode(from_idx)
            to_node = manager.IndexToNode(to_idx)
            return time_matrix[from_node][to_node]

        transit_idx = routing.RegisterTransitCallback(time_callback)
        routing.SetArcCostEvaluatorOfAllVehicles(transit_idx)
        
        # Thêm ràng buộc thời gian (Max 12 tiếng/ngày)
        routing.AddDimension(transit_idx, 30, int(max_time_minutes), True, "Time")
        
        # Cho phép bỏ điểm (Penalty) nếu không kịp giờ
        for node in range(1, n):
            routing.AddDisjunction([manager.NodeToIndex(node)], 100000)

        # 3. Chạy thuật toán
        search_params = pywrapcp.DefaultRoutingSearchParameters()
        # Chiến lược tham lam (nhanh) để khởi tạo
        search_params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
        # Chiến lược Metaheuristic (thông minh) để tối ưu
        search_params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
        search_params.time_limit.seconds = 2

        solution = routing.SolveWithParameters(search_params)

        if solution:
            route = []
            index = routing.Start(0)
            total_time = 0
            while not routing.IsEnd(index):
                node_idx = manager.IndexToNode(index)
                poi_id = idx_to_id[node_idx]
                
                route.append({
                    "order": len(route) + 1,
                    "poi_id": poi_id,
                    "type": "Start" if len(route)==0 else "Visit"
                })
                
                index = solution.Value(routing.NextVar(index))
                if not routing.IsEnd(index):
                    next_node = manager.IndexToNode(index)
                    total_time += time_matrix[node_idx][next_node]
            
            return {"status": "success", "total_time": total_time, "route": route}
        
        return {"status": "fail"}

In [14]:
import pandas as pd
from collections import defaultdict

class HistoryAnalyzer:
    def __init__(self):
        self.co_occurrence = defaultdict(lambda: defaultdict(int))

    def fit(self, history_csv_path):
        """Học: Người ta thường đi đâu sau điểm A?"""
        try:
            df = pd.read_csv(history_csv_path)
            # Sắp xếp đúng thứ tự chuyến đi
            df = df.sort_values(['itinerary_id', 'day', 'order'])
            
            for _, group in df.groupby(['itinerary_id', 'day']):
                pois = group['poi_id'].tolist()
                for i in range(len(pois) - 1):
                    curr, next_p = pois[i], pois[i+1]
                    self.co_occurrence[curr][next_p] += 1
            print(f"✓ History Patterns Learned: {len(self.co_occurrence)} nodes.")
        except Exception as e:
            print(f"⚠ History load failed: {e}")

    def get_popularity_score(self, current_poi, candidate_poi):
        """Tính điểm phổ biến của cặp (A -> B)"""
        if current_poi not in self.co_occurrence: return 0.0
        
        transitions = self.co_occurrence[current_poi]
        count = transitions.get(candidate_poi, 0)
        total = sum(transitions.values())
        
        return count / total if total > 0 else 0

In [15]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

class TimeAwareMultiDayPlanner:
    def __init__(self, recommender, optimizer, df_poi, history_engine=None):
        self.recommender = recommender
        self.optimizer = optimizer
        self.history_engine = history_engine
        self.poi_df = df_poi.copy()
        if "poi_id" in self.poi_df.columns: self.poi_df = self.poi_df.set_index("poi_id")

    def _get_type_group(self, poi_type):
        t = str(poi_type).lower()
        if t in ['restaurant', 'bar/pub', 'vegetarian', 'bakery', 'food']: return 'food'
        elif t in ['coffee shop', 'cafe', 'tea shop']: return 'cafe'
        else: return 'attraction'

    def _find_smart_next_poi(self, current_id, candidate_df):
        if candidate_df.empty: return None
        try:
            curr_lat = self.poi_df.loc[current_id, 'latitude']
            curr_lon = self.poi_df.loc[current_id, 'longitude']
        except: return candidate_df.iloc[0]['poi_id']

        dists = np.sqrt((candidate_df['latitude'] - curr_lat)**2 + (candidate_df['longitude'] - curr_lon)**2)
        dist_score = 1 / (dists + 1e-5)
        dist_score = dist_score / dist_score.max()

        hist_scores = np.zeros(len(candidate_df))
        if self.history_engine:
            for i, cand_id in enumerate(candidate_df['poi_id']):
                hist_scores[i] = self.history_engine.get_popularity_score(current_id, cand_id)

        final_scores = (0.7 * dist_score) + (0.3 * hist_scores)
        return candidate_df.iloc[np.argmax(final_scores)]['poi_id']

    def plan_itinerary(self, user_profile, total_days, start_poi_id):
        # 1. LẤY GỢI Ý & GOM NHÓM (Code cũ...)
        all_scored = self.recommender.recommend(
            self.poi_df.reset_index(), user_profile['user_city'], 
            user_profile['user_type'], user_profile['user_price'], top_k=200
        )
        if all_scored.empty: return []
        
        all_scored['group'] = all_scored['poi_type'].apply(self._get_type_group)
        pool_attr = all_scored[all_scored['group'] == 'attraction'].copy().head(total_days * 10)
        pool_food = all_scored[all_scored['group'] == 'food'].copy()
        pool_cafe = all_scored[all_scored['group'] == 'cafe'].copy()

        k = min(total_days, len(pool_attr))
        if k == 0: return []
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        pool_attr['day_cluster'] = kmeans.fit_predict(pool_attr[['latitude', 'longitude']].values)

        full_schedule = []
        unique_clusters = sorted(pool_attr['day_cluster'].unique())
        
        for day_idx, cluster_id in enumerate(unique_clusters):
            if day_idx >= total_days: break
            day_attr_ids = pool_attr[pool_attr['day_cluster'] == cluster_id]['poi_id'].tolist()
            if not day_attr_ids: continue

            opt_res = self.optimizer.optimize_route(
                day_attr_ids, start_poi_id, max_time_minutes=840, visit_time_per_poi=45
            )
            if opt_res['status'] != 'success': continue
            
            skeleton = opt_res['route']
            final_day_route = []
            
            current_time = 8.0 
            has_lunch, has_cafe = False, False
            
            prev_poi_id = None # Lưu điểm trước để tính travel time
            
            for i, step in enumerate(skeleton):
                poi_id = step['poi_id']
                
                # --- TÍNH THỜI GIAN DI CHUYỂN TỪ ĐIỂM TRƯỚC ---
                travel_min = 0
                if prev_poi_id:
                    # Gọi hàm get_time từ optimizer
                    travel_min = self.optimizer.get_time_between(prev_poi_id, poi_id)
                
                prev_poi_id = poi_id # Cập nhật điểm hiện tại làm điểm trước cho vòng sau
                
                # Cộng thời gian di chuyển vào current_time
                current_time += (travel_min / 60.0) 

                if i == 0: # Điểm xuất phát
                    step['travel_time'] = 0
                    step['arrival_time'] = "08:00"
                    final_day_route.append(step)
                    continue

                # Add Attraction
                step['travel_time'] = travel_min
                step['arrival_time'] = f"{int(current_time):02d}:{int((current_time%1)*60):02d}"
                
                if 'name' not in step: 
                    step['name'] = self.poi_df.loc[poi_id, 'name'] if poi_id in self.poi_df.index else poi_id
                
                final_day_route.append(step)
                current_time += 1.0 # 1h Visit
                
                # --- LOGIC ĂN TRƯA ---
                if 11.5 <= current_time <= 13.5 and not has_lunch:
                    lunch_id = self._find_smart_next_poi(poi_id, pool_food)
                    if lunch_id:
                        # Tính travel time từ điểm tham quan -> quán ăn
                        t_lunch = self.optimizer.get_time_between(poi_id, lunch_id)
                        current_time += (t_lunch / 60.0)
                        
                        final_day_route.append({
                            "order": -1, "poi_id": lunch_id, "type": "Lunch",
                            "name": self.poi_df.loc[lunch_id, 'name'],
                            "arrival_time": f"{int(current_time):02d}:{int((current_time%1)*60):02d}",
                            "travel_time": t_lunch
                        })
                        pool_food = pool_food[pool_food['poi_id'] != lunch_id]
                        has_lunch = True
                        current_time += 1.0 # Ăn 1h
                        prev_poi_id = lunch_id # Update prev để tính tiếp

                # --- LOGIC CAFE ---
                elif 15.0 <= current_time <= 16.5 and not has_cafe:
                    cafe_id = self._find_smart_next_poi(poi_id, pool_cafe)
                    if cafe_id:
                        t_cafe = self.optimizer.get_time_between(poi_id, cafe_id)
                        current_time += (t_cafe / 60.0)
                        
                        final_day_route.append({
                            "order": -1, "poi_id": cafe_id, "type": "Coffee",
                            "name": self.poi_df.loc[cafe_id, 'name'],
                            "arrival_time": f"{int(current_time):02d}:{int((current_time%1)*60):02d}",
                            "travel_time": t_cafe
                        })
                        pool_cafe = pool_cafe[pool_cafe['poi_id'] != cafe_id]
                        has_cafe = True
                        current_time += 0.75 # Cafe 45p
                        prev_poi_id = cafe_id

            for idx, item in enumerate(final_day_route): item['order'] = idx + 1
            full_schedule.append({"day": day_idx + 1, "route": final_day_route})

        return full_schedule

    def export_csv(self, schedule):
        rows = []
        for d in schedule:
            for step in d['route']:
                rows.append({
                    "day": f"day{d['day']}", 
                    "time": step.get('arrival_time', ''),
                    "travel_min": step.get('travel_time', 0), # Cột mới
                    "poi_name": step.get('name', ''), 
                    "type": step.get('type', 'Visit'),
                    "poi_id": step['poi_id']
                })
        return pd.DataFrame(rows)

In [16]:
from preprocess import load_and_clean_data
PATH = {
    "poi": '../data/POI.csv',
    "dist": '../data/distance_km.csv',
    "dur": '../data/duration_min.csv',
    "history": '../data/itinerary_history.csv',
    "model_rec": '../models/catboost_rec.pkl'
}


print(">>> Loading Data...")
df_poi = load_and_clean_data("../data/POI.csv")

print(">>> Initializing Engines...")
rec_engine = CatBoostRecommender()
if os.path.exists(PATH["model_rec"]):
    rec_engine.load(PATH["model_rec"])
else:
    print("Training Model...")
    df_train = pd.read_csv(PATH["train"])
    rec_engine.train(df_train)
    rec_engine.save(PATH["model_rec"])

opt_engine = RouteOptimizer(PATH["dur"], PATH["dist"], PATH["poi"])
hist_engine = HistoryAnalyzer()
if os.path.exists(PATH["history"]): hist_engine.fit(PATH["history"])

planner = TimeAwareMultiDayPlanner(rec_engine, opt_engine, df_poi, hist_engine)

base_model = CatBoostRecommender()
base_model.load("../models/catboost_recommender.pkl")

# USER INPUT GIẢ LẬP
req = {"user_city": "ha noi", "user_type": "entertainment", "user_price": 1}

# --- SỬA ĐOẠN NÀY ---
# 1. Lọc POI theo thành phố user yêu cầu (Hà Nội)
city_key = req['user_city'].lower().strip()
city_pois = df_poi[df_poi['city_norm'] == city_key]

if city_pois.empty:
    print(f"❌ Lỗi: Không tìm thấy dữ liệu cho thành phố '{city_key}'")
    start_id = "unknown"
else:
    # 2. Tìm khách sạn TRONG thành phố đó
    available_hotels = city_pois[city_pois['type'].str.contains('hotel', case=False, na=False)]
    
    if not available_hotels.empty:
        start_id = available_hotels.iloc[0]['poi_id']
        hotel_name = available_hotels.iloc[0]['name']
        print(f"✓ Chọn điểm xuất phát: {hotel_name} ({start_id})")
    else:
        # Nếu không có khách sạn, lấy điểm đầu tiên của thành phố làm điểm Start
        start_id = city_pois.iloc[0]['poi_id']
        print(f"⚠ Không có khách sạn, xuất phát từ: {city_pois.iloc[0]['name']}")

print(f"\n>>> Planning Trip: {req}")
schedule = planner.plan_itinerary(req, total_days=3, start_poi_id=start_id)

print(f"\n>>> Planning Trip: {req}")
schedule = planner.plan_itinerary(req, total_days=3, start_poi_id=start_id)

if schedule:
    df_res = planner.export_csv(schedule)
    print("\n=== LỊCH TRÌNH ===")
    print(df_res)
    df_res.to_csv("my_trip_result.csv", index=False)
else:
    print("Failed to plan.")

>>> Loading Data...
>>> Initializing Engines...
✓ [CatBoost] Model loaded.
✓ History Patterns Learned: 2540 nodes.
✓ [CatBoost] Model loaded.
✓ Chọn điểm xuất phát: Wyndham Garden Hanoi (hotel01843)

>>> Planning Trip: {'user_city': 'ha noi', 'user_type': 'entertainment', 'user_price': 1}


d:\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



>>> Planning Trip: {'user_city': 'ha noi', 'user_type': 'entertainment', 'user_price': 1}


d:\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



=== LỊCH TRÌNH ===
     day   time  travel_min  \
0   day1  08:00           0   
1   day1  08:13          14   
2   day1  09:17           4   
3   day1  10:22           5   
4   day1  11:37          15   
5   day1  12:54          16   
6   day1  14:09          15   
7   day1  15:22          13   
8   day1  16:17          10   
9   day2  08:00           0   
10  day2  08:11          11   
11  day2  09:13           2   
12  day2  10:13           0   
13  day2  11:13           0   
14  day2  12:14           1   
15  day2  13:17           3   
16  day2  14:26           9   
17  day2  15:34           8   
18  day2  16:19           0   
19  day2  17:20           1   
20  day2  18:22           2   
21  day2  19:23           1   
22  day2  20:24           1   
23  day2  21:25           1   
24  day2  22:26           1   
25  day2  23:30           5   
26  day2  24:35           5   
27  day3  08:00           0   
28  day3  08:16          16   
29  day3  09:22           6   
30  day3  10:23    